# 09 -- Re-rate headline checkpoints with full records (plan v5 Step 12e)

Ratings from before Step 12 kept only the summary row (Elo, CI, games). This notebook re-rates the 7 headline
checkpoints (Derek, Nov 1) the same way they were first rated -- same opponents, 20 games each, same openings --
but now every game is saved to `results/games.csv` and each row gets wins/draws/losses. The new row replaces
the old one; its note keeps the old rating. Games against deterministic opponents may replay exactly; games
against time-limited Stockfish won't, so a rating can shift a little.

**Resumable:** each row is saved after every 10-game batch, and a checkpoint that already has its full set of
saved games is skipped on a rerun. **Needs a GPU pod** (the LLM players). ~6 h: ~1 h per LLM checkpoint,
~30-45 min per AlphaZero one; the LLM ones run first.

In [ ]:
# Setup (RunPod): the repo lives on the volume; artifacts under artifact_root (configs/base.yaml).
import os
_here = os.getcwd()
REPO_DIR = os.path.dirname(_here) if os.path.basename(_here) == "notebooks" else _here
%cd {REPO_DIR}
!git log -1 --oneline
!pip install -q -r requirements.txt

import yaml
import torch
ROOT = yaml.safe_load(open("configs/base.yaml"))["artifact_root"]
from engine.stockfish import ensure_stockfish, SF_VERSION
print("Stockfish:", SF_VERSION, ensure_stockfish())
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none -- this needs a GPU pod"
print("GPU:", GPU)

## The checkpoints
Each spec: its `results.csv` row name, how to load it (`eval.agents.load_player`), and the opponents it was
first rated against. `dataset_size` fills that column where it applies (labeled positions trained on).

In [ ]:
LLM_OPPONENTS = ["greedy", "step_20000", "sf_1320"]      # as in notebooks 02 and 04
AZ_OPPONENTS = ["random", "greedy", "step_20000"]         # as in notebook 05

RL_RUNS = ["structured_masked_sft25000_s0",                      # run 1, defaults
           "structured_masked_sft25000_s0_sfend",                # run 2, Stockfish-scored endings
           "structured_masked_sft25000_s0_curric120",            # run 3, curriculum
           "structured_masked_sft25000_s0_curric120_lr5x"]       # run 4, curriculum + 5x learning rate

SPECS = (
    [{"name": "structured_masked_nall_e10_s0@15000", "kind": "sft", "run": "structured_masked_nall_e10_s0",
      "step": 15000, "dataset_size": 44982, "opponents": LLM_OPPONENTS}]
    + [{"name": f"rl_{run}@r20", "kind": "llm_rl", "run": run, "step": 20, "opponents": LLM_OPPONENTS}
       for run in RL_RUNS]
    + [{"name": f"az_b6c64_s0@r{r}", "kind": "alphazero", "run": "az_b6c64_s0", "step": r, "simulations": 50,
        "opponents": AZ_OPPONENTS} for r in (5, 53)]
)

from eval.anchors import ResultsTable
eval_cfg = yaml.safe_load(open("configs/eval.yaml"))
GAMES, BATCH = eval_cfg["games_per_opponent"], eval_cfg["batch_games"]
results = ResultsTable(os.path.join(ROOT, eval_cfg["results_file"]))
missing = [s["name"] for s in SPECS if results.role(s["name"]) != "model"]
assert not missing, f"no model row for {missing} -- check the names"
for s in SPECS:
    r = results.rows[s["name"]]
    wdl = f"{r['wins']}/{r['draws']}/{r['losses']}" if r["wins"] != "" else "-"
    print(f"{s['name']:<60} now {r['rating']} [{r['ci_low']}, {r['ci_high']}], {r['games']} games, W/D/L {wdl}")

## Re-rate
One checkpoint at a time; opponents are built once per opponent set.

In [ ]:
from eval.agents import is_rerated, rerate
from eval.anchors import build_reference_agents, close_agents
from eval.elo import format_table

for opp_names in (LLM_OPPONENTS, AZ_OPPONENTS):
    todo = [s for s in SPECS if s["opponents"] == opp_names
            and not is_rerated(results, s["name"], GAMES * len(opp_names))]
    for s in SPECS:
        if s["opponents"] == opp_names and s not in todo:
            print(f"{s['name']}: already re-rated with records -- skipped")
    if not todo:
        continue
    opponents = build_reference_agents(results, opp_names, ROOT)
    try:
        for s in todo:
            print(f"\n== {s['name']} vs {opp_names}")
            est = rerate(s, results, ROOT, opponents, games_per_opponent=GAMES, batch_games=BATCH, gpu=GPU)
            print(format_table([est]))
            torch.cuda.empty_cache()
    finally:
        close_agents(opponents)

## Before and after
New ratings with W/D/L, the old rating (from the note), and how each checkpoint's games ended.

In [ ]:
import pandas as pd

t = ResultsTable(results.path)
rows = pd.DataFrame([t.rows[s["name"]] for s in SPECS])
display(rows[["name", "rating", "ci_low", "ci_high", "games", "wins", "draws", "losses", "sec_per_move", "note"]])
games = pd.DataFrame([g for s in SPECS for g in t.games(s["name"])])
if len(games):
    display(games.groupby(["subject", "termination"]).size().unstack(fill_value=0))